# Traffic Condition Classification with PyTorch (CE 738)

A multi-class classifier that predicts **traffic class (0, 1, 2)** from five traffic and weather features using a single-layer neural network (softmax regression) in PyTorch.

**Requirements:** `pip install pandas scikit-learn torch`

## 1. Load the data
The dataset (`transportation.csv`) contains 30 records with 7 columns: a record ID, five input features, and the target `traffic_class`.

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── 1. Read data ──
df = pd.read_csv("transportation.csv"  # place the CSV in the same folder as this notebook)
print(f"Shape: {df.shape}")
df.head()

Shape: (30, 7)


,record_id,traffic_volume_vph,occupancy_pct,average_speed_mph,rainfall_mm,travel_time_min,traffic_class
0,1,320,7,64,0,7.6,0
1,2,410,9,62,0,8.0,0
2,3,500,11,60,1,8.5,0
3,4,590,13,58,0,9.0,0
4,5,680,15,56,2,9.7,0


## 2. Preprocess
`record_id` is removed because it is an identifier, not a predictor. The data is then checked for missing values (none were found).

In [8]:
# ── 2. Preprocess ──
# Drop record_id (not a feature)
df = df.drop(columns=['record_id'])

# Check for nulls
print(f"\nNull counts:\n{df.isnull().sum()}")


Null counts:
traffic_volume_vph    0
occupancy_pct         0
average_speed_mph     0
rainfall_mm           0
travel_time_min       0
traffic_class         0
dtype: int64


## 3. Define inputs and target
- **Features (X):** traffic volume (veh/h), occupancy (%), average speed (mph), rainfall (mm), travel time (min)
- **Target (y):** `traffic_class` — 3 classes with 10 records each (balanced)

In [9]:
# ── 3. Define input and target variables ──
X = df.drop(columns=['traffic_class'])  # Features: traffic_volume_vph, occupancy_pct, average_speed_mph, rainfall_mm, travel_time_min
y = df['traffic_class']                 # Target: 0, 1, 2

print(f"\nFeatures: {list(X.columns)}")
print(f"Target classes: {sorted(y.unique())}")
print(f"Class distribution:\n{y.value_counts().sort_index()}")


Features: ['traffic_volume_vph', 'occupancy_pct', 'average_speed_mph', 'rainfall_mm', 'travel_time_min']
Target classes: [np.int64(0), np.int64(1), np.int64(2)]
Class distribution:
traffic_class
0    10
1    10
2    10
Name: count, dtype: int64


## 4. Train-test split
An 80/20 split gives 24 training and 6 test records. `stratify=y` keeps the class proportions equal in both sets, and `random_state=42` makes the split reproducible.

In [10]:
# ── 4. Train-test split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")


Train: 24 samples | Test: 6 samples


## 5. Scale features
Features are on very different scales (e.g., volume in hundreds vs. rainfall in single digits), so they are standardized to mean 0 and standard deviation 1. The scaler is fit on the training set only to avoid data leakage.

In [11]:
# ── 5. Scale features ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train and evaluate the model
- Data is converted to PyTorch tensors and loaded in mini-batches of 8.
- **Model:** one linear layer mapping 5 inputs to 3 class scores (logits) — equivalent to multinomial logistic regression.
- **Loss:** `CrossEntropyLoss`, which applies softmax internally and suits multi-class classification.
- **Optimizer:** stochastic gradient descent, learning rate 0.03, 10 epochs.
- **Prediction:** the class with the highest score (`argmax`).

**Result:** 83.33% test accuracy (5 of 6 test records correct).

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)  # reproducible results

# ── Convert to tensors ──
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

# ── DataLoader ──
train_iter = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=8, shuffle=True)

# ── Model, loss, optimizer ──
d = X_train_t.shape[1]  # 5 features
net = nn.Sequential(nn.Linear(d, 3))  # 3 output classes instead of 1
loss = nn.CrossEntropyLoss()           # classification loss instead of MSE
opt = torch.optim.SGD(net.parameters(), lr=0.03)

# ── Training loop ──
num_epochs = 10
for epoch in range(num_epochs):
    for X_batch, y_batch in train_iter:
        opt.zero_grad()
        loss(net(X_batch), y_batch).backward()
        opt.step()

# ── Evaluate ──
with torch.no_grad():
    preds = net(X_test_t).argmax(dim=1)
    acc = (preds == y_test_t).float().mean()
    print(f"Test Accuracy: {acc:.2%}")

Test Accuracy: 83.33%


## Notes and limitations
- The dataset is very small (30 records), so the 6-record test set gives a rough accuracy estimate; one misclassification changes accuracy by about 17 percentage points. Cross-validation would give a more reliable estimate.
- Accuracy may differ slightly between runs of the training cell because of weight initialization and batch shuffling; `torch.manual_seed(42)` is set to reduce this.